# Prep

## Import stuff

In [ ]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math
import ast
import os

## Some magical magic to make the R stuff work

In [ ]:
# The rpy2/R session and CFA machinery now live in the hitop_cfa package.
# Importing it starts the embedded R session: it sets the BLAS threading
# env vars (if unset) and loads base, utils, lavaan, and the patched semTools.
import hitop_cfa
from hitop_cfa.r_env import (ro, rbase, utils, lavaan, semtools,
                             RRuntimeError, pandas2ri)

import rpy2.ipython.html
rpy2.ipython.html.init_printing()

# Paths

In [ ]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'manual_temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'


## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [ ]:
import platform


def get_architecture():
    # Get the raw machine architecture string
    arch = platform.machine().lower()
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    elif "x86" in arch or "amd" in arch or "i386" in arch or "i686" in arch:
        return "x86"
    else:
        return f"Unknown ({arch})"


total_cpus = os.cpu_count()
# account for hyperthreading
arch = get_architecture()
if arch == 'ARM':
    cpus_to_use = total_cpus - 2
else:
    cpus_to_use = total_cpus // 2 - 1
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000

## SET SEEDS !!!!!!!!!!

In [ ]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [ ]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

### TEST THE SEEDS!!!!!!!!

In [ ]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

In [ ]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [ ]:
from hitop_cfa import (
    build_luts,
    check_hitop_ids,
    cfa_helper_func,
    run_specific_cfa,
    do_three_way_cfa_stepwise_mi,
    do_three_way_cfa_stepwise_scalar,
    exhaustive_cfa_ablations,
    set_seeds,
    silence_r,
)

# item-text lookups (was load_item_lookup + inline lut construction)
_luts = build_luts(path_to_item_lookup, path_to_cogmood_questions)
item_lookup = _luts['item_lookup']
item_lut = _luts['item_lut']
phq_lut = _luts['phq_lut']
gad_lut = _luts['gad_lut']
baars_lut = _luts['baars_lut']

## CFA wrapper functions

# Run Main Code

## Load preprocessed data and concatinate

In [ ]:
# load
data_val = pd.read_csv(path_save_val)
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# concatinate
data_val_genpop = pd.concat([data_val, data_gp])
data_val_enriched = pd.concat([data_val, data_en])
data_genpop_enriched = pd.concat([data_gp, data_en])

# Dataset pairs for the hitop_cfa functions. Unified on the full
# genpop+enriched source (the old norecontact variant was column-identical
# on every CFA item, so this cannot change results).
datasets = {'val_gp': data_val_genpop, 'val_en': data_val_enriched, 'gp_en': data_genpop_enriched}
datasets_gp_en = {'gp_en': data_genpop_enriched}

# Do main CFA analysis

### 1. Find invariant subsets via exhaustive search

#### Looping through all scales where the full item set fails the scalar target:

All models go through the measEq / Wu–Estabrook ladder (configural → thresholds → metric → scalar → strict; see `HANDOFF_measeq_fix.md`), and the exhaustive search targets **scalar**: a combination counts as a success only when the requested level's permutation p ≥ .05 on every pair (the pre-measEq code gated on metric even for scalar searches). The `noninvariant_scales` list is **derived** from the measEq baseline results (`orig_cfa_res.csv`, written by `NB_2_cfa_as_reg.ipynb` — run that baseline first): scales whose full item set already reaches scalar on all pairs are skipped.

In [ ]:
orig_items = {
    'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
    'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop240 + hitop248 + hitop265',
    'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
    'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
    'cognitive_problems': 'cognitive_problems =~hitop67 + hitop159 + hitop189 + hitop142',
    'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
    'indecisiveness': 'indecisiveness =~hitop21 + hitop90 + hitop95',
    'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
    'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
    'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
    'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
    'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
    'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
    'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'
}

In [ ]:
# Scales needing an exhaustive subset search, DERIVED from the measEq
# baseline (run the NB_2_cfa_as_reg baseline first; it writes
# orig_cfa_res.csv). The search targets SCALAR, so a scale needs it unless
# every pair's pscalar is numeric and >= .05 (levels never reached are 'NA'
# strings -> to_numeric coerces to NaN -> counts as failing).
orig_cfa_res = pd.read_csv(cfa_dir / 'orig_cfa_res.csv')
pscalar_num = pd.to_numeric(orig_cfa_res['pscalar'], errors='coerce')
scalar_pass_by_scale = pscalar_num.ge(0.05).groupby(orig_cfa_res['scale']).all()
noninvariant_scales = [s for s in orig_items
                       if not bool(scalar_pass_by_scale.get(s, False))]
print(f"{len(noninvariant_scales)} of {len(orig_items)} scales fail 3-way "
      f"scalar invariance on the full item set:")
print(noninvariant_scales)

lookingforinv_log = log_dir / 'mylog_3wayCFA_lookingforinv_seed12345_v1.txt'
with lookingforinv_log.open('w') as f:
    with redirect_stdout(f):
        for scale in noninvariant_scales:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = orig_items[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa -- whichcfa='scalar': combinations must pass the
                # full ladder through SCALAR on every pair to count as
                # successes (matches the prereg's scalar-core target; note
                # the old code gated success on metric even here)
                successful_combinations_for_scale = exhaustive_cfa_ablations(
                    whichscale=scale,
                    whichcfa='scalar',
                    howmanyitems=how_many_to_try,
                    orig_items=orig_items,
                    datasets=datasets,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                )
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale:
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

In [ ]:
lfi_out = lookingforinv_log.read_text().split('\n')



In [ ]:
lfi_parsing = []
for lix, line in enumerate(lfi_out):
    if line.startswith('!!!!!'):
        row= dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        lfi_parsing.append(row)


In [ ]:
ds_pairs = ['val_gp', 'val_en', 'gp_en']
# measEq / Wu-Estabrook ladder: cfa_helper_func now reports a thresholds
# level between config and metric
inv_levels = ['config', 'thresholds', 'metric', 'scalar', 'strict']

In [ ]:
inv_dat = []
for lprow in lfi_parsing:
    scale_stats = ast.literal_eval(lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        ogformula = orig_items[lprow['scale']]
        itemformula_only = ogformula.split("=~",1)[1]
        ogitems = itemformula_only.split(" + ")
        row = lprow.copy()
        item_nos = iss[0]
        items = [item_lut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [item_lut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        inv_dat.append(row)
inv_dat = pd.DataFrame(inv_dat)
if inv_dat.empty:
    # no exhaustive successes at all: keep the schema so the review-note
    # merge and pickle cells below still run instead of crashing
    inv_dat = pd.DataFrame(columns=['lix', 'scale', 'n_items', 'n_removed',
                                    'ssix', 'item_nos', 'removed_nos',
                                    'items', 'removed'])

In [ ]:
inv_dat

In [ ]:
for row in inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for kk, ii in zip(row.removed_nos, row.removed):
        print(kk, ':', ii)
    print()
    print()

In [ ]:
review_notes = [
    {
        'scale': 'anhedonic_depression',
        'ssix': 4,
        'note': '2 energy items and one anhedonic item exluded',
        'stepwise_agrees':True
    },
    {
        'scale': 'anxious_worry',
        'ssix': 2,
        'note': 'excluded items are more somatic',
    },
    {
        'scale': 'appetite_gain',
        'ssix': 0,
        'note': 'this solution exludes the only item that refers to a thought',
        'stepwise_agrees':True
    },
    {
        'scale': 'hyposomnia',
        'ssix': 0, 
        'note': "stepwise solution found by exhaustive, might be more likely to be interpretted as related to exercise than the others",
        'stepwise_agrees': True
    },
    {
        'scale': 'panic',
        'ssix': 1,
        'note': "scale is just the the physical symptoms of panic attack, arbitrary to pick one, but trembling or shaking was found by the stepwise",
        'stepwise_agrees': True
    },
    {
        'scale': 'separation_insecurity', 
        'ssix': 0,
        'note': "could not handle rejection is an ambiguous item, what does failing to handle mean here",
        'stepwise_agrees': False
    },
    {
        'scale': 'shame_guilt',
        'ssix': 0 ,
        'note': 'may indicate that the scale is misnamed or that the other items imply an excess where feeling guilty could be interpretted as normal',
        'stepwise_agrees': True
    },
    {
        'scale': 'situational_phobia',
        'ssix': 1,
        'note': 'avoiding riding in elevators is not writen as "I was afraid"',
        'stepwise_agrees': False
    },
    {
        'scale': 'social_anxiety', 
        'ssix': 0,
        'note': "stepwise bug, not a great interpretation, they're the only two items that mention people",
        'stepwise_agrees': False
    },
    {
        'scale': 'well_being',
        'ssix': 0,
        'note': "removed items are actions, retained are all 'I felt'",
        'stepwise_agrees': False
    }
]
review_notes = pd.DataFrame(review_notes)
review_notes['exhaustive_choice'] = True


In [ ]:
review_notes

In [ ]:
inv_dat = inv_dat.merge(review_notes, how='outer', on=['scale', 'ssix'])
inv_dat['exhaustive_choice'] = inv_dat.exhaustive_choice.fillna(False)

In [ ]:
inv_dat.to_pickle(cfa_dir / 'exhaustive.pkl')

In [ ]:
inv_dat.loc[inv_dat.exhaustive_choice]

### 3. (if needed) Check invariance of any excluded subsets that have 3 or more items

The subsets are **derived** from the final-cores table (`stepwise_scalar.pkl`, written by the scalar-continuation loop in `NB_2_cfa_as_reg.ipynb`): for every scale whose final core dropped 3+ items, the removed items are tested as their own scale up the full ladder.

In [ ]:
# Excluded subsets with >= 3 items, derived from the final cores
# (previously a hardcoded dict of outcome-specific formulas)
final_cores = pd.read_pickle(cfa_dir / 'stepwise_scalar.pkl')
to_test_inv = {}
for row in final_cores.itertuples():
    removed = getattr(row, 'removed_nos', None)
    if isinstance(removed, list) and len(removed) >= 3:
        to_test_inv[row.scale] = f"{row.scale} =~" + " + ".join(removed)
print(f"{len(to_test_inv)} excluded subset(s) with >= 3 items:")
print(to_test_inv)

with open(log_dir / "mylog_3wayCFA_invsubsets_seed12345_2.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test
            run_specific_cfa(
                whichscale=scale,
                item_list=items_list,
                whichcfa='strict',
                datasets=datasets,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
            )

## For interpretation - check what items mean

In [ ]:
# Item-id lists per scale, generated from orig_items (the old hardcoded
# dict had drifted: it was missing indecisiveness entirely and dropped
# some items, e.g. 240 from anxious_worry and 159 from cognitive_problems)
scales = {
    f'items_{scale}': [ii.strip().replace('hitop', '')
                       for ii in formula.split('=~', 1)[1].split(' + ')]
    for scale, formula in orig_items.items()
}

for scale, items in scales.items():
    print(scale)
    check_hitop_ids(items, my_item_lookup=item_lookup)

# Analysis for BAARS, GAD, PHQ

In [ ]:
other_scales = {
    'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
    'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
    'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
    'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
    'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
    'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}

# item-text lookup per other scale (keyed by scale name so downstream cells
# no longer depend on a hardcoded, order-aligned list)
other_luts = {
    'phq_sum': phq_lut,
    'gad_sum': gad_lut,
    'baars_inattention_sum': baars_lut['inattention'],
    'baars_hyperactivity_sum': baars_lut['hyperactivity'],
    'baars_impulsivity_sum': baars_lut['impulsivity'],
    'baars_sct_sum': baars_lut['sct'],
}

other_log = log_dir /'mylog_CFA_baarsgadphq_seed12345.txt'
with other_log.open("w") as f:
    with redirect_stdout(f):
        other_cfa_res = []
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")

            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")

            # +++ GENPOP VS ENRICHED +++
            print('\n -----> GENPOP VS ENRICHED <----- ')
            (flag_metric, pconfig, pthresholds, pmetric,
             pscalar, pstrict) = cfa_helper_func(
                            scalename=scale,
                            list_of_items=items_list,
                            mydata_python=data_genpop_enriched,
                            mydata_temp_path=path_to_helpfile,
                            num_iter=num_iter,
                            cpus_to_use=cpus_to_use,
                            max_level='strict')
            other_cfa_res.append(dict(
                pair='gp_en', scale=scale, pconfig=pconfig,
                pthresholds=pthresholds, pmetric=pmetric,
                pscalar=pscalar, pstrict=pstrict))
other_cfa_res = pd.DataFrame(other_cfa_res)
other_cfa_res.to_csv(cfa_dir / 'other_orig_cfa_res.csv', index=None)
other_cfa_res

In [ ]:
# Other scales needing an exhaustive subset search, DERIVED from the
# other-scales baseline above: a scale needs it unless its gp_en pscalar
# is numeric and >= .05 (these searches are single-pair, gp_en only).
other_pscalar = pd.to_numeric(other_cfa_res['pscalar'], errors='coerce')
other_scalar_pass = other_pscalar.ge(0.05).groupby(other_cfa_res['scale']).all()
other_failing_scales = [s for s in other_scales
                        if not bool(other_scalar_pass.get(s, False))]
print(f"{len(other_failing_scales)} of {len(other_scales)} other scales "
      f"fail scalar invariance on the full item set:")
print(other_failing_scales)

other_exhaustive_logs = log_dir /'mylog_exhuastive_inv_search_baarsgadphq_seed12345.txt'

with other_exhaustive_logs.open("w") as f:
    with redirect_stdout(f):
        for scale in other_failing_scales:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = other_scales[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = exhaustive_cfa_ablations(
                    whichscale=scale,
                    whichcfa='scalar',
                    howmanyitems=how_many_to_try,
                    orig_items=other_scales,
                    datasets=datasets_gp_en,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                )
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale:
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

In [ ]:
other_lfi_out = other_exhaustive_logs.read_text().split('\n')
other_lfi_parsing = []
for lix, line in enumerate(other_lfi_out):
    if line.startswith('!!!!!'):
        row = dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        other_lfi_parsing.append(row)
other_ds_pairs = ['gp_en']

In [ ]:
other_inv_dat = []
for lprow in other_lfi_parsing:
    # item-text lookup resolved by scale name (previously a hardcoded list
    # of luts that had to stay order-aligned with a hardcoded scale list)
    olut = other_luts[lprow['scale']]
    scale_stats = ast.literal_eval(other_lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        row = lprow.copy()
        item_nos = iss[0]
        ogitems = list(olut.keys())
        items = [olut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [olut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in other_ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        other_inv_dat.append(row)
other_inv_dat = pd.DataFrame(other_inv_dat)
if other_inv_dat.empty:
    # no exhaustive successes: keep the schema so downstream cells run
    other_inv_dat = pd.DataFrame(columns=['lix', 'scale', 'n_items',
                                          'n_removed', 'ssix', 'item_nos',
                                          'removed_nos', 'items', 'removed'])

In [ ]:
for row in other_inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for kk, ii in zip(row.removed_nos, row.removed):
        print(kk, ':', ii)
    print()
    print()

In [ ]:
# gonna try inattention dropping 1, 8, and 9
other_scales = {
    'baars_inattention_sum': 'baars_inattention_sum =~inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7',
}

other_log = log_dir /'mylog_inattention_drop_3_seed12345.txt'
with other_log.open("w") as f:
    with redirect_stdout(f):
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")

            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")

            # +++ GENPOP VS ENRICHED +++
            print('\n -----> GENPOP VS ENRICHED <----- ')
            (flag_metric_gp_en, pconfig_gp_en, pthresholds_gp_en, pmetric_gp_en,
             pscalar_gp_en, pstrict_gp_en) = cfa_helper_func(
                            scalename=scale,
                            list_of_items=items_list,
                            mydata_python=data_genpop_enriched,
                            mydata_temp_path=path_to_helpfile,
                            num_iter=num_iter,
                            cpus_to_use=cpus_to_use,
                            max_level='strict')

In [ ]:
# nan-tolerant conversion: untested levels come back as the string 'NA'
# (float('NA') would crash the notebook mid-run if any level failed above)
def _naf(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


inattention_drop_3_manual_row = {
    'scale': 'baars_inattention_sum',
    'n_items': 6,
    'n_removed': 3,
    'ssix':3,
    'item_nos': (
        'inattention_2',
        'inattention_3',
        'inattention_4',
        'inattention_5',
        'inattention_6',
        'inattention_7'
    ),
    'removed_nos': [
        'inattention_1',
        'inattention_8',
        'inattention_9'
    ],
    'items': [
        'Difficulty sustaining my attention in tasks or fun activities',
        "Don't listen when spoken to directly",
        "Don't follow through on instructions and fail to finish work or chores",
        'Have difficulty organizing tasks and activities',
        'Avoid, dislike, or am reluctant to engage in tasks that require sustained mental effort',
        'Lose things necessary for tasks or activities'
    ],
    'removed': [
        'Fail to give close attention to details or make careless mistakes in my work or other activities',
        'Easily distracted by extraneous stimuli or irrelevant thoughts',
        'Forgetful in daily activities'
    ],
    'gp_en__config': _naf(pconfig_gp_en),
    'gp_en__thresholds': _naf(pthresholds_gp_en),
    'gp_en__metric': _naf(pmetric_gp_en),
    'gp_en__scalar': _naf(pscalar_gp_en),
    'gp_en__strict': _naf(pstrict_gp_en)
}
other_inv_dat = pd.concat([other_inv_dat, pd.DataFrame(pd.Series(inattention_drop_3_manual_row)).T])


In [ ]:
other_review_notes = [
    {
        'scale': 'phq_sum',
        'ssix': 0,
        'note': 'Only solution with 2 items removed, may be items with different interpretations in clinical and non-clincial populations',
    },
    {
        'scale': 'gad_sum',
        'ssix': 0,
        'note': 'Annoyance or irritability is potentialy a different construct, but may also be particularly susceptible to different interpretations in clinical and non-clincial populations',
    },
    {
        'scale': 'baars_inattention_sum',
        'ssix': 3,
        'note': 'Dropping all three of the problematic items is invariant and I think more consistent. They all could be differently interpretted.',
    },
    {
        'scale': 'baars_sct_sum',
        'ssix': 1,
        'note': 'These two seem like the most easily misintepretted by non-clinical populations.',
    },
]
other_review_notes = pd.DataFrame(other_review_notes)
other_review_notes['exhaustive_choice'] = True


In [ ]:
other_inv_dat = other_inv_dat.merge(other_review_notes, how='outer', on=['scale', 'ssix'])
other_inv_dat['exhaustive_choice'] = other_inv_dat.exhaustive_choice.fillna(False)


In [ ]:
other_inv_dat.to_pickle(cfa_dir / 'other_scales_exhaustive.pkl')